# w1-hab · Привычки и здоровье

Наблюдательный анализ связи четырёх блоков привычек — **питание, сон, стресс,
физическая активность** — с ожирением (ИМТ ≥ 30).

- Основная выборка: **NHANES 2017–2018** (измеренные ИМТ, давление, пульс;
  24-часовой дневник; сон; PHQ-9 как прокси стресса; активность).
- Внешние проверки: **BRFSS 2024** и **AI4FoodDB**.

Ноутбук показывает шаги и объясняет «зачем»; финальные артефакты собирает
`python analysis.py`. Все выводы — ассоциации, не причинность.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import sys

import numpy as np
import pandas as pd

ROOT = next(
    p
    for p in (Path.cwd(), Path.cwd() / "works" / "w1-hab")
    if (p / "data" / "PASSPORT.md").exists()
)
sys.path.insert(0, str(ROOT))
OUT = ROOT / "out"
RAW = ROOT / "data" / "raw"

passport = (ROOT / "data" / "PASSPORT.md").read_text(encoding="utf-8")
print(passport[:700])

## 1. Паспорт до расчётов

Пока нет ссылки, лицензии и sha256, анализ не начинаем: иначе неизвестно,
что именно мы моделируем. Сверяем хеши всех файлов из паспорта с диском.


In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

rows = re.findall(r"\|\s*`([^`]+)`\s*\|\s*[\d,\s]+\|\s*`([0-9a-f]{64})`\s*\|", passport)
results = []
for rel, expected in rows:
    path = ROOT / rel
    actual = sha256(path) if path.exists() else "missing"
    results.append((rel, actual == expected))
    print(f"{rel:<50} {'OK' if actual == expected else 'MISMATCH'}")
assert results and all(ok for _, ok in results), "паспорт не сходится с файлами"
print(f"проверено записей: {len(results)}, все совпали")

## 2. Очистка без заполнения средним

Заполнять средним нельзя: это сужает дисперсию и создаёт ложную уверенность.
Стратегия — **complete-case для модели** плюс явные правила для невозможных
значений. Ниже — шаги и потери по каждому шагу.


In [ ]:
import analysis as A

prep = A.prepare_nhanes()
steps = pd.DataFrame(prep["steps"])
display(steps.style.format({"n": "{:,}", "removed": "{:,}"}))
print("в модель вошло:", len(prep["model"]), "из", prep["n_raw"])

In [ ]:
missing = pd.DataFrame(prep["missing"])
outliers = pd.DataFrame(prep["outliers"])
display(missing)
display(outliers)

## 3. Целевая переменная

`obesity = BMXBMI >= 30`. ИМТ измерен, а не самоотчёт (в отличие от BRFSS),
поэтому это более надёжный исход. ИМТ не входит в признаки — утечки нет.


In [ ]:
df = prep["model"]
print(df["BMXBMI"].describe().round(2))
print(df["obesity"].value_counts(normalize=True).round(3).to_string())

## 4. Baseline

Константное предсказание — нижняя граница: любая модель должна быть лучше
доли положительного класса. ROC AUC baseline = 0,5 по построению.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X = df[prep["features"]]
y = df["obesity"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
baseline = DummyClassifier(strategy="prior").fit(Xtr, ytr)
print("baseline AUC:", round(roc_auc_score(yte, baseline.predict_proba(Xte)[:, 1]), 3))
print("test n:", len(yte), "events:", int(yte.sum()))

## 5. Логистическая регрессия

Масштабирование и one-hot — **внутри Pipeline**, fit только на train:
иначе средние и категории теста подсказывают модели ответ. Тест не трогаем
до оценки.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pre = ColumnTransformer(
    [
        ("num", StandardScaler(), prep["num"]),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), prep["cat"]),
    ],
    remainder="drop",
)
pipe = Pipeline(
    [("pre", pre), ("lr", LogisticRegression(max_iter=2000, random_state=42))]
).fit(Xtr, ytr)
p_model = pipe.predict_proba(Xte)[:, 1]
print("test AUC:", round(roc_auc_score(yte, p_model), 3))

## 6. Permutation importance

Перемешиваем признак на **тесте** и смотрим падение ROC AUC: насколько модель
теряет, если признак перестаёт нести информацию. 50 повторов здесь — демо;
в `analysis.py` — 1000 повторов и отдельные интервалы для блоков.


In [ ]:
from sklearn.inspection import permutation_importance

r = permutation_importance(
    pipe, Xte, yte, scoring="roc_auc", n_repeats=50, random_state=42
)
imp = pd.Series(r.importances_mean, index=prep["features"]).sort_values(ascending=False)
display(imp.head(10).to_frame("Δ ROC AUC"))

## 7. Факторы: направление, сила, 95% CI

OR на 1 SD (числовые после `log1p`) или уровень против референса.
Направление определяется по 95% CI: если интервал включает 1 — `unclear`,
такой фактор не выдаём за значимый. CI — bootstrap, 1000 ресэмплов.


In [ ]:
factors = pd.read_csv(OUT / "factors.csv", encoding="utf-8-sig")
cols = ["factor", "block", "or_per_sd", "ci_low", "ci_high", "direction", "strength", "perm_importance"]
display(factors[cols].head(15).round(3))

## 8. Традиционный способ

До ML ту же задачу решали таблицами сопряжённости: доли по группам, простой
OR и χ². Это прозрачно, но без поправок на конфаундеры. Счёт из четырёх
флагов показывает, сколько даёт «ручная» сумма признаков.


In [ ]:
metrics = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
trad = metrics["traditional"]
display(pd.DataFrame(trad["rows"]).round(3))
display(pd.DataFrame(trad["score"]))
print("AUC счёта:", round(trad["auc_score"], 3))
print("логит на флагах (CV):", round(trad["auc_flags_cv"], 3))
print("полная модель (CV):", round(trad["auc_full_cv"], 3))

## 9. Ограничения и вывод

- Наблюдательные данные: связи не доказывают причинность; возможны
  конфаундеры и обратная причинность.
- NHANES: один 24-ч дневник, самоотчёт о сне, PHQ-9 — прокси стресса.
- Complete-case теряет строки; выборка смещена к полным наблюдениям.
- Калории и вода могут отражать недоучёт и размер тела — их OR не «эффект».
- BRFSS и AI4FoodDB — независимые выборки, не соединяются с NHANES.

Итог: питание — точечно самый сильный блок, но его CI пересекается с CI
сна и активности, поэтому ранг между блоками не устойчив. Стресс-прокси
даёт слабую положительную связь, но permutation-вклад блока неотличим от нуля.
